In [9]:
import os
import pandas as pd
import numpy as np
import re 
import matplotlib.pyplot as plt
import seaborn as sns
import calendar
from datetime import datetime, timedelta
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.colors import ListedColormap
import geopandas as gpd
from shapely.validation import make_valid

# --- Configuration ---
path_to_scenarios   = '/home/abhis/india_power/scenarios'
path_to_save_images = '/home/abhis/india_power/images'
path_to_images = '/home/abhis/india_power/images'
path_to_csvs        = '/home/abhis/india_power/gridpath_india_viz/csvs'

path_to_data = '/home/abhis/india_power/gridpath_india_viz/data/'
tech_labels_ = pd.read_csv(path_to_csvs + '/technology_labels.csv') 
# scenarios_   = pd.read_csv(path_to_csvs + '/ra-scenario_labels.csv')
scenarios_   = pd.read_csv(path_to_csvs + '/ra-12_prm_no_ccap.csv')


# Load and processing system load timepoints

In [ ]:
# Load and processing system load timepoints
def _processing_system_load_timepoints(scenarios_):

    dfs_ = []
    for scenario, path_to_scenarios in zip(scenarios_['scenario'], scenarios_['path']):
        path_to_scenario = os.path.join(path_to_scenarios, scenario)

        for dirpath, dirnames, filenames in os.walk(path_to_scenario):
            # Processing scenarios results folder only
            if dirpath.split('/')[-1] == 'results':
                print(scenario)

                df_ = pd.read_csv(os.path.join(dirpath, 'system_load_zone_timepoint.csv'), low_memory = False)
                
                df_ = df_[['timepoint', 
                           'load_zone', 
                           'static_load_mw',
                           'net_imports_mw', 
                           'overgeneration_mw', 
                           'unserved_energy_mw']]
                
                df_ = df_.groupby(['timepoint',
                                   'load_zone']).agg({'static_load_mw': 'sum',
                                                      'net_imports_mw': 'sum',
                                                      'overgeneration_mw': 'sum',
                                                      'unserved_energy_mw': 'sum'}).reset_index(drop = False)
                
                df_ = df_.rename(columns = {'static_load_mw': 'static_load',
                                            'net_imports_mw': 'net_imports',
                                            'overgeneration_mw': 'overgeneration',
                                            'unserved_energy_mw': 'unserved_energy'})
                
                # df_ = pd.melt(df_, id_vars = ['timepoint', 
                #                                'load_zone'], 
                #                 value_vars = ['static_load', 
                #                               'net_imports',
                #                               'overgeneration',
                #                               'unserved_energy'], 
                #                 var_name   = 'technology', 
                #                 value_name = 'power_mw')
                
                # Add iteration descriptors to dataframe
                df_['W'] = dirpath.split('/')[-5].split('_')[-1]
                df_['H'] = dirpath.split('/')[-4].split('_')[-1]
                df_['A'] = dirpath.split('/')[-3].split('_')[-1]
                
                dfs_.append(df_)

    return pd.concat(dfs_).reset_index(drop = True)

df_load_ = _processing_system_load_timepoints(scenarios_)
print(df_load_.head())

pier_12_prm_no_ccap_2040_1
pier_12_prm_no_ccap_2040_1
pier_12_prm_no_ccap_2040_1
pier_12_prm_no_ccap_2040_1
pier_12_prm_no_ccap_2040_1
pier_12_prm_no_ccap_2040_1
pier_12_prm_no_ccap_2040_1
pier_12_prm_no_ccap_2040_1
pier_12_prm_no_ccap_2040_1
pier_12_prm_no_ccap_2040_1
pier_12_prm_no_ccap_2040_1
pier_12_prm_no_ccap_2040_1
pier_12_prm_no_ccap_2040_2
pier_12_prm_no_ccap_2040_2
pier_12_prm_no_ccap_2040_2
pier_12_prm_no_ccap_2040_2
pier_12_prm_no_ccap_2040_2
pier_12_prm_no_ccap_2040_2
pier_12_prm_no_ccap_2040_2
pier_12_prm_no_ccap_2040_2
pier_12_prm_no_ccap_2040_2
pier_12_prm_no_ccap_2040_2
pier_12_prm_no_ccap_2040_2
pier_12_prm_no_ccap_2040_2
pier_12_prm_no_ccap_2040_3
pier_12_prm_no_ccap_2040_3
pier_12_prm_no_ccap_2040_3
pier_12_prm_no_ccap_2040_3
pier_12_prm_no_ccap_2040_3
pier_12_prm_no_ccap_2040_3
pier_12_prm_no_ccap_2040_3
pier_12_prm_no_ccap_2040_3
pier_12_prm_no_ccap_2040_3
pier_12_prm_no_ccap_2040_3
pier_12_prm_no_ccap_2040_3
pier_12_prm_no_ccap_2040_3
pier_12_prm_no_ccap_2040_4
p

In [ ]:
# Create a timestamp column from 'timepoint' field (YYYYMMDDHH)
tp = df_load_['timepoint'].astype(str)
dates = tp.str[:8]
hours = tp.str[8:].astype(int)  #1-24

df_load_['timestamp'] = (
    pd.to_datetime(dates, format='%Y%m%d')
    + pd.to_timedelta(hours, unit='h')
)
# Reorder so timestamp is first
cols = df_load_.columns.tolist()
cols.insert(0, cols.pop(cols.index('timestamp')))
df_load_ = df_load_[cols]

# State-level metrics for all iterations (includes national metrics)

In [ ]:
df = df_load_.copy()

# Exclude Bhutan
df = df[df['load_zone'] != "Bhutan"].copy()

HOURS_PER_YEAR = 8760.0  
tau = 1.0  # 1 MW threshold for LOL event

df['is_lol']     = df['unserved_energy'] > tau
df['date']       = df['timestamp'].dt.date
df['hrs_in_tmp'] = 1

def compute_metrics(sub, label):
    """Compute RA metrics for a given dataframe slice."""
    num_iters = sub[['W','H','A']].drop_duplicates().shape[0]
    hrs_run_total = float(sub['hrs_in_tmp'].sum())
    n_years = hrs_run_total / HOURS_PER_YEAR if hrs_run_total else np.nan

    total_lol_hours    = int(sub['is_lol'].sum())
    total_unserved_MWh = float((sub['unserved_energy'] * sub['hrs_in_tmp']).sum())
    total_energy_MWh   = float((sub['static_load']     * sub['hrs_in_tmp']).sum())

    days_with_lol = int(
        sub.groupby(['W','H','A','date'])['is_lol'].max().sum()
    )

    # Annualized metrics
    EUE_MWh_per_yr   = (total_unserved_MWh / n_years) if np.isfinite(n_years) else np.nan
    LOLH_hrs_per_yr  = (total_lol_hours / n_years) if np.isfinite(n_years) else np.nan
    LOLE_days_per_yr = (days_with_lol / n_years) if np.isfinite(n_years) else np.nan

    return {
        'load_zone':               label,
        'iterations (W,H,A)':      num_iters,
        'LOLH (hrs/yr)':           LOLH_hrs_per_yr,
        'EUE (MWh/yr)':            EUE_MWh_per_yr,
        'LOLE (days/yr)':          LOLE_days_per_yr,
        'LOLE (days/10yr)':        (LOLE_days_per_yr * 10) if np.isfinite(LOLE_days_per_yr) else np.nan,
        'LOLP (fraction of hrs)':  (total_lol_hours / hrs_run_total) if hrs_run_total > 0 else np.nan,
        'LOLP (%)':                ((total_lol_hours / hrs_run_total) * 100) if hrs_run_total > 0 else np.nan,
        'Annual Energy (MWh/yr)':  (total_energy_MWh / n_years) if np.isfinite(n_years) else np.nan,
        'NENS (%)':                (100 * EUE_MWh_per_yr / (total_energy_MWh / n_years))
                                   if (n_years and total_energy_MWh > 0) else np.nan,
    }

# --- per-zone metrics ---
zone_metrics = []
for zone, sub in df.groupby("load_zone"):
    zone_metrics.append(compute_metrics(sub.copy(), zone))
metrics_df = pd.DataFrame(zone_metrics)

# --- national metric (all zones pooled) ---
national_metric_df = pd.DataFrame([
    compute_metrics(df.copy(), "National (excl. Bhutan)")
])

In [ ]:
# --- outputs ---
print("=== Per-zone metrics ===")
print(metrics_df)

=== Per-zone metrics ===
             load_zone  iterations (W,H,A)  LOLH (hrs/yr)   EUE (MWh/yr)  \
0       Andhra_Pradesh                  18       0.722222    3604.699696   
1    Arunachal_Pradesh                  18       0.000000       0.000000   
2                Assam                  18       0.000000       0.000000   
3                Bihar                  18       0.222222     559.541355   
4           Chandigarh                  18       2.000000     114.419374   
5         Chhattisgarh                  18       0.000000       0.000000   
6   Dadra_Nagar_Haveli                  18      31.833333   44064.756921   
7            Daman_Diu                  18      12.444444    9277.222660   
8                Delhi                  18       5.277778   30440.186657   
9                  Goa                  18      16.111111   14022.796991   
10             Gujarat                  18       9.888889   59188.790782   
11             Haryana                  18       0.222222    15

In [ ]:
print("\n=== National metric (excluding Bhutan) ===")
print(national_metric_df)


=== National metric (excluding Bhutan) ===
                 load_zone  iterations (W,H,A)  LOLH (hrs/yr)  EUE (MWh/yr)  \
0  National (excl. Bhutan)                  18       4.171569  13245.006512   

   LOLE (days/yr)  LOLE (days/10yr)  LOLP (fraction of hrs)  LOLP (%)  \
0        0.289216          2.892157                0.000476  0.047621   

   Annual Energy (MWh/yr)  NENS (%)  
0            1.208298e+08  0.010962  
